# CS 4412 - M3 Complete Implementation
**Topic:** Sleep Health and Lifestyle Pattern Discovery

This notebook delivers a runnable M3 pipeline with updated preprocessing, clustering, association rules, PCA, interpretable classification, anomaly detection, and interpretation connected to the project discovery questions.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor

## 2. Load data

In [ ]:
df = pd.read_csv('../data/raw/Sleep_health_and_lifestyle_dataset.csv')
df.head()

## 3. Basic overview and cleaning

In [ ]:
print('Shape before cleaning:', df.shape)
print('\nMissing values:')
print(df.isna().sum())

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['sleep_disorder'] = df['sleep_disorder'].fillna('None')

bp = df['blood_pressure'].str.split('/', expand=True)
df['systolic_bp'] = pd.to_numeric(bp[0], errors='coerce')
df['diastolic_bp'] = pd.to_numeric(bp[1], errors='coerce')
df = df.drop(columns=['blood_pressure'])

print('\nDuplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().copy()
print('Shape after cleaning:', df.shape)
df.head()

## 4. EDA

In [ ]:
numeric_cols = [
    'age','sleep_duration','quality_of_sleep','physical_activity_level',
    'stress_level','heart_rate','daily_steps','systolic_bp','diastolic_bp'
]
categorical_cols = ['gender','occupation','bmi_category','sleep_disorder']

print(df[numeric_cols].describe().round(2))
print('\nCategorical counts:')
for col in categorical_cols:
    print('\n' + col)
    print(df[col].value_counts())

In [ ]:
df[numeric_cols].hist(figsize=(14,10), bins=15)
plt.suptitle('Numeric Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(10,8))
plt.imshow(corr, cmap='coolwarm', interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()
corr.round(2)

In [ ]:
pairs = [
    ('sleep_duration','quality_of_sleep'),
    ('sleep_duration','stress_level'),
    ('physical_activity_level','quality_of_sleep'),
    ('daily_steps','heart_rate')
]
for x_col, y_col in pairs:
    plt.figure(figsize=(6,4))
    plt.scatter(df[x_col], df[y_col], alpha=0.7)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f'{y_col} vs {x_col}')
    plt.show()

## 5. Scaling for clustering

In [ ]:
X_cluster = df[numeric_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)
print(X_scaled.shape)

## 6. Clustering comparison

In [ ]:
k_results = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    k_results.append([k, score])

k_results_df = pd.DataFrame(k_results, columns=['k', 'silhouette_score'])
k_results_df

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(k_results_df['k'], k_results_df['silhouette_score'], marker='o')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.title('K-Means Silhouette Scores')
plt.show()

In [ ]:
best_k = int(k_results_df.sort_values('silhouette_score', ascending=False).iloc[0]['k'])
print('Best k:', best_k)

df['kmeans_cluster'] = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X_scaled)
df['agg_cluster'] = AgglomerativeClustering(n_clusters=best_k).fit_predict(X_scaled)

dbscan = DBSCAN(eps=1.2, min_samples=5)
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

comparison = []
comparison.append(['KMeans', silhouette_score(X_scaled, df['kmeans_cluster'])])
comparison.append(['Agglomerative', silhouette_score(X_scaled, df['agg_cluster'])])

if len(set(df['dbscan_cluster']) - {-1}) > 1:
    mask = df['dbscan_cluster'] != -1
    comparison.append(['DBSCAN', silhouette_score(X_scaled[mask], df.loc[mask, 'dbscan_cluster'])])

comparison_df = pd.DataFrame(comparison, columns=['method', 'silhouette_score']).sort_values('silhouette_score', ascending=False)
comparison_df

## 7. Cluster profiles

In [ ]:
cluster_profile_numeric = df.groupby('kmeans_cluster')[numeric_cols].mean().round(2)
cluster_profile_numeric

In [ ]:
cluster_profile_categorical = pd.DataFrame({
    col: df.groupby('kmeans_cluster')[col].agg(lambda x: x.value_counts().index[0])
    for col in categorical_cols
})
cluster_profile_categorical

## 8. PCA visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['pc1'] = X_pca[:,0]
df['pc2'] = X_pca[:,1]

print('Explained variance ratio:', pca.explained_variance_ratio_)
print('Total explained variance:', pca.explained_variance_ratio_.sum())

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(df['pc1'], df['pc2'], c=df['kmeans_cluster'], alpha=0.8)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Projection with K-Means Clusters')
plt.show()

## 9. Interpretable classification

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[numeric_cols], df['kmeans_cluster'],
    test_size=0.25, random_state=42, stratify=df['kmeans_cluster']
)

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)
tree_preds = tree_model.predict(X_test)

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)

print('Decision Tree accuracy:', (tree_preds == y_test).mean())
print('Naive Bayes accuracy:', (nb_preds == y_test).mean())
print('\nDecision Tree confusion matrix:')
print(confusion_matrix(y_test, tree_preds))
print('\nNaive Bayes confusion matrix:')
print(confusion_matrix(y_test, nb_preds))

In [ ]:
feature_importance = pd.Series(tree_model.feature_importances_, index=numeric_cols).sort_values(ascending=False)
feature_importance

In [ ]:
plt.figure(figsize=(16,8))
plot_tree(
    tree_model,
    feature_names=numeric_cols,
    class_names=[str(x) for x in sorted(df['kmeans_cluster'].unique())],
    filled=True
)
plt.title('Decision Tree Explaining K-Means Clusters')
plt.show()

## 10. Association rules on discretized features

In [ ]:
rules_df = df.copy()
rules_df['sleep_duration_bin'] = pd.cut(rules_df['sleep_duration'], bins=[0,6,7,10], labels=['short_sleep','medium_sleep','long_sleep'], include_lowest=True)
rules_df['stress_bin'] = pd.cut(rules_df['stress_level'], bins=[0,4,7,10], labels=['low_stress','medium_stress','high_stress'], include_lowest=True)
rules_df['steps_bin'] = pd.cut(rules_df['daily_steps'], bins=[0,5000,10000,20000], labels=['low_steps','medium_steps','high_steps'], include_lowest=True)
rules_df['quality_bin'] = pd.cut(rules_df['quality_of_sleep'], bins=[0,4,7,10], labels=['low_quality','medium_quality','high_quality'], include_lowest=True)
rules_df['age_bin'] = pd.cut(rules_df['age'], bins=[0,35,50,100], labels=['young_adult','middle_age','older_adult'], include_lowest=True)

tx_cols = ['gender','occupation','bmi_category','sleep_disorder','sleep_duration_bin','stress_bin','steps_bin','quality_bin','age_bin']
transactions = []
for _, row in rules_df.iterrows():
    transactions.append([f"{col}={row[col]}" for col in tx_cols])

n = len(transactions)
item_counts = Counter()
pair_counts = Counter()

for tx in transactions:
    uniq = sorted(set(tx))
    item_counts.update(uniq)
    for i in range(len(uniq)):
        for j in range(i + 1, len(uniq)):
            pair_counts[(uniq[i], uniq[j])] += 1

rules = []
for (a, b), cnt in pair_counts.items():
    for ant, cons in [(a, b), (b, a)]:
        support = cnt / n
        confidence = cnt / item_counts[ant]
        lift = confidence / (item_counts[cons] / n)
        rules.append([ant, cons, support, confidence, lift, cnt])

association_rules = pd.DataFrame(rules, columns=['antecedent','consequent','support','confidence','lift','count'])
association_rules = association_rules[
    (association_rules['support'] >= 0.08) &
    (association_rules['confidence'] >= 0.55) &
    (association_rules['lift'] > 1.10)
].sort_values(['lift','confidence','support'], ascending=False).reset_index(drop=True)

association_rules.head(15)

## 11. Anomaly detection

In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_labels = lof.fit_predict(X_scaled)
df['lof_flag'] = lof_labels
df['lof_anomaly'] = np.where(lof_labels == -1, 'Anomaly', 'Normal')
df['lof_score'] = lof.negative_outlier_factor_

df['lof_anomaly'].value_counts()

In [ ]:
anomalies = df[df['lof_anomaly'] == 'Anomaly'].copy()
anomalies.head(10)

In [ ]:
colors = df['lof_anomaly'].map({'Normal': 0, 'Anomaly': 1})
plt.figure(figsize=(7,5))
plt.scatter(df['pc1'], df['pc2'], c=colors, alpha=0.8)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Projection with LOF Anomalies')
plt.show()

## 12. Save outputs

In [ ]:
df.to_csv('../outputs/sleep_health_m3_full_results.csv', index=False)
comparison_df.to_csv('../outputs/clustering_method_comparison_m3.csv', index=False)
cluster_profile_numeric.to_csv('../outputs/cluster_profile_numeric_m3.csv')
cluster_profile_categorical.to_csv('../outputs/cluster_profile_categorical_modes_m3.csv')
association_rules.to_csv('../outputs/association_rules_m3.csv', index=False)
anomalies.to_csv('../outputs/lof_anomalies_m3.csv', index=False)

print('Saved M3 output files to ../outputs')

## 13. Interpretation summary

### What the clustering suggests
The clustering comparison shows meaningful structure in the real-world sleep dataset. K-Means and Agglomerative both produce usable segments, while DBSCAN detects denser local groupings and noise points.

### What the association rules suggest
The strongest rules show repeated co-occurrence patterns among occupation, sleep-disorder status, sleep-duration buckets, stress, and BMI. These results are more useful for discovery than prediction because they reveal conditions that tend to appear together.

### What the classification models suggest
The decision tree and Naive Bayes models are used here as interpretability tools. Their high agreement with cluster labels suggests that the discovered cluster structure is strongly reflected in the numeric sleep-health variables.

### What the anomalies suggest
LOF identifies a small group of participants whose profiles differ from the majority. These cases are useful for follow-up because they may represent unusual combinations of sleep quality, steps, stress, and vital signs.

### Discovery question progress
This notebook advances the original project by moving from simulated fatigue data to a real-world sleep-health dataset. It now supports stronger answers about natural groupings, co-occurrence patterns, and unusual profiles.
